In [6]:
DataSet_Path = "/content/drive/MyDrive/dataset/task"

In [7]:
print(DataSet_Path )

/content/drive/MyDrive/dataset/task


In [8]:
import os
import cv2
TRAIN_PATH = os.path.join(DataSet_Path,"train")
TEST_PATH = os.path.join(DataSet_Path,"test")
VALID_PATH = os.path.join(DataSet_Path,"valid")

In [9]:
print(TEST_PATH)

/content/drive/MyDrive/dataset/task/test


In [10]:
IMG_SIZE = 128
import numpy as np

In [11]:
def read_image(image_path):
  img = cv2.imread(image_path,cv2.IMREAD_GRAYSCALE)

  if img is None:
    return None

  #resize
  img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))

  return np.array(img)

In [12]:
def prepare_data(folder_path):
  images = []
  labels = []
  skipped = 0

  class_folders = os.listdir(folder_path)
  print("Classes Found: ",class_folders)

  for class_name in class_folders:
    class_path = os.path.join(folder_path,class_name)
    class_lower = class_name.lower()
    if class_lower == "women":
      label = 0
    elif class_lower == "men":
      label = 1
    else:
      print("Unknown class folder")
      continue
    #images read from folder
    image_files = os.listdir(class_path)
    for img_file in image_files:
      img_path = os.path.join(class_path,img_file)

      if not img_file.lower().endswith((".jpg",".jpeg",".png",".giff")):
        continue

      img = read_image(img_path)
      if img is not None:
        images.append(img)
        labels.append(label)
      else:
        skipped +=1
    print("Loaded", len(os.listdir(class_path)),"images from ", class_name, "Label = ",label)
  return np.array(images),np.array(labels)

In [ ]:
x_train,y_train = prepare_data(TRAIN_PATH)
print(f'Training set: {x_train.shape[0]} images')

Classes Found:  ['women', 'men']


In [ ]:
x_test,y_test = prepare_data(TEST_PATH)
print(f'Test set: {x_test.shape[0]} images')

In [ ]:
x_valid,y_valid = prepare_data(VALID_PATH)
print(f'Validation set: {x_valid.shape[0]} images')

In [ ]:
import matplotlib.pyplot as plt

# Find some male and female examples from training data
female_indices = np.where(y_train == 0)[0][:5]   # First 5 female images
male_indices   = np.where(y_train == 1)[0][:5]   # First 5 male images

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Sample Training Images\n(Row 1: Female | Row 2: Male)",
             fontsize=14, fontweight='bold')

# Row 1: Female images
for i, idx in enumerate(female_indices):
    axes[0, i].imshow(x_train[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    axes[0, i].set_title(f"Female (0)", fontsize=9, color='deeppink')
    axes[0, i].axis('off')

# Row 2: Male images
for i, idx in enumerate(male_indices):
    axes[1, i].imshow(x_train[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    axes[1, i].set_title(f"Male (1)", fontsize=9, color='royalblue')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#flatten data means coverting 2D image to 1D array
x_train = x_train.reshape(len(x_train), IMG_SIZE * IMG_SIZE)
x_test = x_test.reshape(len(x_test), IMG_SIZE * IMG_SIZE)
x_valid = x_valid.reshape(len(x_valid), IMG_SIZE * IMG_SIZE)

print(x_train.shape[0])

In [ ]:
#normalize
from sklearn import preprocessing

# Ensure data is 2D before normalization
x_train = x_train.reshape(len(x_train), IMG_SIZE * IMG_SIZE)
x_test = x_test.reshape(len(x_test), IMG_SIZE * IMG_SIZE)
x_valid = x_valid.reshape(len(x_valid), IMG_SIZE * IMG_SIZE)

x_train = preprocessing.normalize(x_train)
x_test = preprocessing.normalize(x_test)
x_valid = preprocessing.normalize(x_valid)

In [ ]:
#random shuffle
from sklearn.utils import shuffle
x_train, y_train = shuffle(x_train,y_train,random_state=42)
x_test,y_test = shuffle(x_test,y_test,random_state=42)
x_valid, y_valid = shuffle(x_valid,y_valid, random_state=42)

In [ ]:
print("After Shuffle")
print(y_train[:20])

In [ ]:
#one-hot encode label
y_train_int = y_train.copy()
y_test_int = y_test.copy()
y_valid = y_valid.copy()

In [ ]:
from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test,num_classes=2)
y_valid = to_categorical(y_valid, num_classes=2)

In [ ]:
print("after one-hot encoding")
print("y_train shape",y_train.shape)
print(y_train[y_train==0][:10])
print(y_train[y_train==1][:10])
print(y_train[1])

In [ ]:
# ============================================================
# STEP 1: IMPORT ALL LIBRARIES
# ============================================================

import numpy as np                          # Numerical operations & array handling
import pandas as pd                         # Data manipulation (used for reports)
import matplotlib.pyplot as plt             # Plotting graphs and showing images
import matplotlib.image as mpimg            # Loading images for display
import cv2                                  # OpenCV — image reading and processing
import os                                   # Operating system — browse folders/files
import warnings
warnings.filterwarnings('ignore')           # Suppress minor warnings for cleaner output

# TensorFlow & Keras — the deep learning framework
import tensorflow as tf
import tensorflow.keras as ka
from tensorflow.keras.models import Sequential, model_from_json
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

# Scikit-learn — data preprocessing and evaluation tools
from sklearn import preprocessing
from sklearn.utils import shuffle
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

# PIL — for image handling in live test
from PIL import Image

# Fix random seed for reproducibility
# This ensures we get the same results every run
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully! ✅")

In [ ]:

model = Sequential(name="Gender_detect",layers=[
    Input(shape=(IMG_SIZE*IMG_SIZE,)),
    Dense(128,activation='relu'),
    Dense(64,activation='relu'),
    Dense(2,activation='softmax')
])

print("Model Architecture")
model.summary()

In [ ]:
#compile and train the model

model.compile(
    optimizer= ka.optimizers.Adam(learning_rate=0.001),
    loss = 'categorical_crossentropy',
    metrics= ['accuracy']
)


In [ ]:
history = model.fit(
    x_train,y_train,
    epochs = 50,
    batch_size = 32,
    validation_data=(x_valid,y_valid),
    verbose =1

)

In [ ]:
#evaluate
test_loss,test_accuracy = model.evaluate(x_test,y_test,verbose=0)

print("Test Accuracy: ",test_accuracy)
print("Test Loss: ",test_loss)

In [ ]:
#model
save_path = "/content/drive/MyDrive/dataset/my_model.keras"
model.save(save_path)